In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

In [3]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)

In [4]:
df=pd.read_csv(r"C:\Users\acann\OneDrive\Desktop\mtech major project\Anwesha Chakraborty\output\Final_Dataset_ERA5.csv")

In [5]:
df["Datetime"] = pd.to_datetime(df["Datetime"])

In [6]:
print(df.shape)
print(df.head())
print(df.dtypes)

(21689, 7)
             Datetime  PM2.5_Mean  Visibility  Temperature         RH  \
0 2024-01-01 00:00:00  170.053667    1207.008     8.909912  94.370200   
1 2024-01-01 01:00:00  169.284333    1207.008     8.911804  93.757260   
2 2024-01-01 02:00:00  165.553000    1207.008     8.978394  93.705830   
3 2024-01-01 03:00:00  163.278667    1207.008     8.673157  94.920586   
4 2024-01-01 04:00:00  155.996000    1207.008     8.577515  96.242320   

   WindSpeed         BLH  
0   1.548904   39.382355  
1   1.346162   47.932030  
2   1.031514   57.352417  
3   0.757781   62.579464  
4   0.567124  135.569460  
Datetime       datetime64[ns]
PM2.5_Mean            float64
Visibility            float64
Temperature           float64
RH                    float64
WindSpeed             float64
BLH                   float64
dtype: object


In [7]:
print(df.isnull().sum())

Datetime       0
PM2.5_Mean     0
Visibility     0
Temperature    0
RH             0
WindSpeed      0
BLH            0
dtype: int64


In [8]:
features = [
    "PM2.5_Mean",
    "RH",
    "Temperature",
    "WindSpeed",
    "BLH"
]

target = "Visibility"

In [9]:
df["Visibility_1h"] = df[target].shift(-1)
df["Visibility_3h"] = df[target].shift(-3)
df["Visibility_6h"] = df[target].shift(-6)

In [10]:
df["Target_Time_1h"] = df["Datetime"].shift(-1)
df["Target_Time_3h"] = df["Datetime"].shift(-3)
df["Target_Time_6h"] = df["Datetime"].shift(-6)

In [11]:
df_forecast = df.dropna(
    subset=[
        "Visibility_1h",
        "Visibility_3h",
        "Visibility_6h"
    ]
).copy()
print(df_forecast.shape)

(21683, 13)


In [12]:
fog_months = [1, 2, 3, 11, 12]

case1_train_mask = (
    (df_forecast["Datetime"].dt.year.isin([2024, 2025])) &
    (df_forecast["Datetime"].dt.month.isin(fog_months)))

In [13]:
case2_train_mask = (
    df_forecast["Datetime"].dt.year.isin([2024, 2025]))

In [14]:
def get_test_mask(df, lead_time):

    target_time_col = f"Target_Time_{lead_time}h"

    test_mask = (
        (df[target_time_col] >= "2026-01-01") &
        (df[target_time_col] < "2026-04-01")
    )

    return test_mask

In [15]:
def get_models():

    models = {

        "Linear Regression": {
            "model": LinearRegression(),
            "scale": True
        },

        "Ridge Regression": {
            "model": Ridge(alpha=1),
            "scale": True
        },

        "Random Forest": {
            "model": RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            ),
            "scale": False
        },

        "Gradient Boosting": {
            "model": GradientBoostingRegressor(
                n_estimators=300,
                random_state=42
            ),
            "scale": False
        },

        "SVR": {
            "model": SVR(
                kernel="rbf",
                C=100,
                epsilon=0.1
            ),
            "scale": True
        }
    }

    return models

In [16]:
def run_forecast(
    df,
    model,
    scale_features,
    features,
    lead_time,
    train_mask,
    test_mask
):

    target_column = f"Visibility_{lead_time}h"

    X_train = df.loc[
        train_mask,
        features
    ].copy()

    y_train = df.loc[
        train_mask,
        target_column
    ].copy()

    X_test = df.loc[
        test_mask,
        features
    ].copy()

    y_test = df.loc[
        test_mask,
        target_column
    ].copy()


    train_data = pd.concat(
        [X_train, y_train],
        axis=1
    ).dropna()

    test_data = pd.concat(
        [X_test, y_test],
        axis=1
    ).dropna()


    X_train = train_data[features]
    y_train = train_data[target_column]

    X_test = test_data[features]
    y_test = test_data[target_column]


    if scale_features:

        scaler = StandardScaler()

        X_train_model = scaler.fit_transform(
            X_train
        )

        X_test_model = scaler.transform(
            X_test
        )

    else:

        X_train_model = X_train
        X_test_model = X_test

    model.fit(
        X_train_model,
        y_train
    )

    y_pred = model.predict(
        X_test_model
    )

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    mse = mean_squared_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        y_pred
    )
    return {
        "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2, "Actual": y_test, "Predicted": y_pred
    }

In [17]:
lead_times = [1, 3, 6]

In [18]:
training_cases = {
    "Case 1 - Fog Season": case1_train_mask,
    "Case 2 - All Months": case2_train_mask
}


In [19]:
models = get_models()
all_forecast_results = []
all_forecast_predictions = {}

In [20]:
for case_name, train_mask in training_cases.items():

    for lead_time in lead_times:

        # Test mask for this lead time
        test_mask = get_test_mask(
            df_forecast,
            lead_time
        )
        for model_name, model_info in models.items():
            print(
                f"Running: "
                f"{model_name} | "
                f"{case_name} | "
                f"{lead_time}-hour forecast"
            )
            result = run_forecast(
                df=df_forecast,
                model=model_info["model"],
                scale_features=model_info["scale"],
                features=features,
                lead_time=lead_time,
                train_mask=train_mask,
                test_mask=test_mask
            )
            # Store numerical results
            all_forecast_results.append({
                "Model": model_name,
                "Training Case": case_name,
                "Lead Time (hours)": lead_time,
                "MAE": result["MAE"],
                "MSE": result["MSE"],
                "RMSE": result["RMSE"],
                "R2": result["R2"]
            })
            # Store predictions
            prediction_key = (
                f"{model_name}_"
                f"{case_name}_"
                f"{lead_time}h"
            )
            all_forecast_predictions[
                prediction_key
            ] = {
                "Actual": result["Actual"],
                "Predicted": result["Predicted"]
            }

Running: Linear Regression | Case 1 - Fog Season | 1-hour forecast
Running: Ridge Regression | Case 1 - Fog Season | 1-hour forecast
Running: Random Forest | Case 1 - Fog Season | 1-hour forecast
Running: Gradient Boosting | Case 1 - Fog Season | 1-hour forecast
Running: SVR | Case 1 - Fog Season | 1-hour forecast
Running: Linear Regression | Case 1 - Fog Season | 3-hour forecast
Running: Ridge Regression | Case 1 - Fog Season | 3-hour forecast
Running: Random Forest | Case 1 - Fog Season | 3-hour forecast
Running: Gradient Boosting | Case 1 - Fog Season | 3-hour forecast
Running: SVR | Case 1 - Fog Season | 3-hour forecast
Running: Linear Regression | Case 1 - Fog Season | 6-hour forecast
Running: Ridge Regression | Case 1 - Fog Season | 6-hour forecast
Running: Random Forest | Case 1 - Fog Season | 6-hour forecast
Running: Gradient Boosting | Case 1 - Fog Season | 6-hour forecast
Running: SVR | Case 1 - Fog Season | 6-hour forecast
Running: Linear Regression | Case 2 - All Months | 1

In [22]:
 forecast_results_df = pd.DataFrame(
    all_forecast_results
)
forecast_results_df = forecast_results_df.round(3)
forecast_results_df

,Model,Training Case,Lead Time (hours),MAE,MSE,RMSE,R2
0,Linear Regression,Case 1 - Fog Season,1,806.935,993467.771,996.729,0.577
1,Ridge Regression,Case 1 - Fog Season,1,806.947,993480.888,996.735,0.577
2,Random Forest,Case 1 - Fog Season,1,738.506,875092.055,935.464,0.627
3,Gradient Boosting,Case 1 - Fog Season,1,736.185,871128.607,933.343,0.629
4,SVR,Case 1 - Fog Season,1,728.534,853516.949,923.860,0.637
5,Linear Regression,Case 1 - Fog Season,3,791.851,1028792.498,1014.294,0.562
6,Ridge Regression,Case 1 - Fog Season,3,791.852,1028791.473,1014.294,0.562
7,Random Forest,Case 1 - Fog Season,3,744.478,917468.765,957.846,0.609
8,Gradient Boosting,Case 1 - Fog Season,3,736.609,897388.519,947.306,0.618
9,SVR,Case 1 - Fog Season,3,747.608,937175.121,968.078,0.601


In [23]:
comparison_table = forecast_results_df.pivot_table(
    index=["Model", "Training Case"],
    columns="Lead Time (hours)",
    values=["MAE", "RMSE", "R2"]
)

comparison_table = comparison_table.swaplevel(0, 1, axis=1)
comparison_table = comparison_table.reindex(
    columns=pd.MultiIndex.from_product(
        [[1, 3, 6], ["MAE", "RMSE", "R2"]]
    )
)

comparison_table.round(3)

1                         3  \
                                           MAE      RMSE     R2      MAE   
Model             Training Case                                            
Gradient Boosting Case 1 - Fog Season  736.185   933.343  0.629  736.609   
                  Case 2 - All Months  756.536   950.506  0.615  753.686   
Linear Regression Case 1 - Fog Season  806.935   996.729  0.577  791.851   
                  Case 2 - All Months  837.308  1022.589  0.555  824.452   
Random Forest     Case 1 - Fog Season  738.506   935.464  0.627  744.478   
                  Case 2 - All Months  759.544   955.148  0.612  751.078   
Ridge Regression  Case 1 - Fog Season  806.947   996.735  0.577  791.852   
                  Case 2 - All Months  837.314  1022.598  0.555  824.457   
SVR               Case 1 - Fog Season  728.534   923.860  0.637  747.608   
                  Case 2 - All Months  744.150   944.713  0.620  751.653   

                                                              6            \
                                           RMSE     R2      MAE      RMSE   
Model             Training Case                                             
Gradient Boosting Case 1 - Fog Season   947.306  0.618  881.270  1182.531   
                  Case 2 - All Months   965.963  0.603  859.305  1162.096   
Linear Regression Case 1 - Fog Season  1014.294  0.562  954.855  1284.479   
                  Case 2 - All Months  1046.587  0.534  956.716  1259.411   
Random Forest     Case 1 - Fog Season   957.846  0.609  889.995  1191.046   
                  Case 2 - All Months   965.667  0.603  874.985  1180.286   
Ridge Regression  Case 1 - Fog Season  1014.294  0.562  954.854  1284.470   
                  Case 2 - All Months  1046.592  0.534  956.719  1259.409   
SVR               Case 1 - Fog Season   968.078  0.601  949.111  1293.339   
                  Case 2 - All Months   982.604  0.589  919.521  1268.038   

                                              
                                          R2  
Model             Training Case               
Gradient Boosting Case 1 - Fog Season  0.405  
                  Case 2 - All Months  0.425  
Linear Regression Case 1 - Fog Season  0.298  
                  Case 2 - All Months  0.325  
Random Forest     Case 1 - Fog Season  0.396  
                  Case 2 - All Months  0.407  
Ridge Regression  Case 1 - Fog Season  0.298  
                  Case 2 - All Months  0.325  
SVR               Case 1 - Fog Season  0.288  
                  Case 2 - All Months  0.316

In [24]:
best_models = (
    forecast_results_df
    .loc[
        forecast_results_df
        .groupby(
            [
                "Training Case",
                "Lead Time (hours)"
            ]
        )["MAE"]
        .idxmin()
    ]
    .sort_values(
        [
            "Training Case",
            "Lead Time (hours)"
        ]
    )
)
best_models

,Model,Training Case,Lead Time (hours),MAE,MSE,RMSE,R2
4,SVR,Case 1 - Fog Season,1,728.534,853516.949,923.860,0.637
8,Gradient Boosting,Case 1 - Fog Season,3,736.609,897388.519,947.306,0.618
13,Gradient Boosting,Case 1 - Fog Season,6,881.270,1398379.268,1182.531,0.405
19,SVR,Case 2 - All Months,1,744.150,892482.641,944.713,0.620
22,Random Forest,Case 2 - All Months,3,751.078,932512.063,965.667,0.603
28,Gradient Boosting,Case 2 - All Months,6,859.305,1350468.262,1162.096,0.425
